# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

## Rules

This notebook is checked automatically by **ThetaGrader (TG)**.

- Keep every `### Task N` heading intact -- TG uses it as a marker to locate tasks. Do not remove, rename, renumber, or duplicate these headings.
- Write your solution in the cell(s) reserved for it, directly below the corresponding task statement. Do not move a solution to a different place in the notebook or merge several tasks' solutions together.
- If a task asks for a written conclusion, report, or answer to a theoretical question, put it in a markdown cell directly under that task, clearly separated from code.
- Before submitting your final solution, make sure the `Status` field in the Bot-check cell below is set to `final`.

## AI usage disclosure

If you used AI assistants (e.g. ChatGPT, Claude, GitHub Copilot) while working on this homework, briefly describe here which tool(s) you used, for which parts of the assignment, and how.

Leave this cell as "no AI assistance used" if you did not use any.

In [1]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 47.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 9.2 MB/s eta 0:00:00
  Created wheel for deepspeed: filename=deepspeed-0.19.8-py3-none-any.whl size=2168950 sha256=8bbd0e267f601f99969d795f81ef563cc9de2e316b3b930e60f9f8636c59e33e
  Stored in directory: /root/.cache/pip/wheels/48/df/05/f063123bd6b0c5bf2bdf85f606017f5e3b564c3938673959cd
Successfully built deepspeed
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: datas

### Data Preparation

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

Repo card metadata block was not found. Setting CardData to empty.




Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [11]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [12]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [13]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

In [14]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [15]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [16]:
for batch in val_loader:
    break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

### Task 1 (1 point):

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)

In [18]:
from tqdm.auto import tqdm

correct = 0
total = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

with torch.no_grad():
    progress = tqdm(val_loader, desc="Валидация")

    for batch in progress:
        batch = {key: value.to(device) for key, value in batch.items()}

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            token_type_ids=batch["token_type_ids"],
        )
        predictions = outputs.logits.argmax(dim=1)

        correct += (predictions == batch["labels"]).sum().item()
        total += batch["labels"].size(0)

        progress.set_postfix(accuracy=f"{correct / total:.4f}")

accuracy = correct / total
print(f"Validation accuracy: {accuracy:.4f}")

Валидация:   0%|          | 0/40430 [00:00<?, ?it/s]

Validation accuracy: 0.9084


In [ ]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

### Task 2 (4 points):
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

In [26]:
# Если зависимости уже установлены, повторная установка ничего не изменит
%pip -q install accelerate sentencepiece protobuf

import gc
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(42)

if not torch.cuda.is_available():
    raise RuntimeError("Включи GPU: Среда выполнения → Сменить среду выполнения → GPU")

# Освобождаем GPU от BERT из Task 1
if "model" in globals():
    model.to("cpu")
gc.collect()
torch.cuda.empty_cache()

# 1. Исходные тексты, а не токены BERT
qqp_raw = load_dataset("nyu-mll/glue", "qqp")

# Убираем записи без одного из вопросов
qqp_raw = qqp_raw.filter(
    lambda example: bool(example["question1"] and example["question2"])
)

# 2. Токенизатор именно выбранной модели
MODEL_NAME = "microsoft/deberta-v3-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,
)

def tokenize_pairs(batch):
    return deberta_tokenizer(
        batch["question1"],
        batch["question2"],
        truncation=True,
        max_length=128,
        # Padding сделает collator отдельно для каждого батча
    )

train_data = qqp_raw["train"].map(
    tokenize_pairs,
    batched=True,
    remove_columns=["question1", "question2", "idx"],
)

val_data = qqp_raw["validation"].map(
    tokenize_pairs,
    batched=True,
    remove_columns=["question1", "question2", "idx"],
)

# Trainer ожидает столбец labels
train_data = train_data.rename_column("label", "labels")
val_data = val_data.rename_column("label", "labels")

# 3. Предобученный body + новая head для двух классов
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    dtype=torch.float32,
)

deberta_model.gradient_checkpointing_disable()

# Явно разрешаем обучение всех параметров
for parameter in deberta_model.parameters():
    parameter.requires_grad_(True)

# Экономим память GPU
deberta_model.gradient_checkpointing_disable()

# 4. Метрика
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": float(np.mean(predictions == labels))}

# Сохраняем только номера классов, а не все logits
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

def compute_metrics_from_classes(eval_pred):
    predictions, labels = eval_pred
    return {"accuracy": float(np.mean(predictions == labels))}

# 5. Настройки обучения
training_args = TrainingArguments(
    output_dir="/content/deberta_qqp_checkpoints",
    num_train_epochs=2,
    learning_rate=2e-5,

    # per_device_train_batch_size=8,
    # gradient_accumulation_steps=4,  # эффективный батч: 8 × 4 = 32
    # per_device_eval_batch_size=16,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,  # эффективный батч остаётся 32
    per_device_eval_batch_size=8,

    weight_decay=0.01,
    warmup_steps=190,
    max_grad_norm=1.0,
    fp16=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=1,

    logging_steps=100,
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
    gradient_checkpointing=False,
)

train_small = train_data.shuffle(seed=42).select(
    range(min(30_000, len(train_data)))
)

trainer = Trainer(
    model=deberta_model,
    args=training_args,
    # train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=DataCollatorWithPadding(
        tokenizer=deberta_tokenizer,
        pad_to_multiple_of=8,
    ),
    compute_metrics=compute_metrics_from_classes,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    train_dataset=train_small,
)

# 6. Обучение — прогресс отображается автоматически
trainer.train()

# Здесь уже используется лучшая модель по validation accuracy
metrics = trainer.evaluate()
deberta_accuracy = metrics["eval_accuracy"]

print(f"DeBERTa validation accuracy: {deberta_accuracy:.4f}")

# accuracy из Task 1 не перезаписываем
if "accuracy" in globals():
    print(f"BERT validation accuracy:    {accuracy:.4f}")
    print(f"Разница: {deberta_accuracy - accuracy:+.4f}")

# 7. Сохраняем веса и токенизатор в среде Colab
trainer.save_model("/content/deberta_qqp_final")
deberta_tokenizer.save_pretrained("/content/deberta_qqp_final")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

Epoch,Training Loss,Validation Loss,Accuracy
1,2.485542,0.284824,0.877517
2,1.692877,0.288934,0.883601


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
1.692877,0.288934,2,0.883601


DeBERTa validation accuracy: 0.8836
BERT validation accuracy:    0.9084
Разница: -0.0248


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/deberta_qqp_final/tokenizer_config.json',
 '/content/deberta_qqp_final/tokenizer.json')

### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

### Task 3 (1 point):
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [28]:
import pandas as pd

# После обучения Trainer загрузил лучшую модель
duplicate_model = trainer.model
device = next(duplicate_model.parameters()).device
duplicate_model.eval()

# Собираем уникальные вопросы из обоих столбцов train
train_questions = list(dict.fromkeys(
    question.strip()
    for column in ["question1", "question2"]
    for question in qqp_raw["train"][column]
    if question and question.strip()
))

print("Уникальных вопросов:", len(train_questions))


@torch.inference_mode()
def find_duplicates(question, top_k=5, batch_size=32):
    question = question.strip()
    if not question:
        raise ValueError("Введи непустой вопрос")

    # Исключаем буквальное совпадение с запросом
    candidates = [
        text for text in train_questions
        if text.casefold() != question.casefold()
    ]

    best_scores = np.empty(0, dtype=np.float32)
    best_questions = []

    for start in tqdm(
        range(0, len(candidates), batch_size),
        desc="Поиск дубликатов",
    ):
        texts = candidates[start:start + batch_size]

        inputs = deberta_tokenizer(
            [question] * len(texts),
            texts,
            padding=True,
            truncation=True,
            max_length=128,  # как при обучении
            return_tensors="pt",
        ).to(device)

        logits = duplicate_model(**inputs).logits
        scores = logits.float().softmax(dim=-1)[:, 1].cpu().numpy()

        # Храним только top_k, а не все результаты
        merged_scores = np.concatenate([best_scores, scores])
        merged_questions = best_questions + texts
        indices = np.argsort(merged_scores)[-top_k:][::-1]

        best_scores = merged_scores[indices]
        best_questions = [merged_questions[i] for i in indices]

    return pd.DataFrame({
        "question": best_questions,
        "duplicate_score": best_scores,
    })


# Выбираем пять разных запросов с положительной разметкой
examples = []
seen = set()

for row in qqp_raw["validation"]:
    query = row["question1"]

    if row["label"] == 1 and query.casefold() not in seen:
        examples.append(row)
        seen.add(query.casefold())

    if len(examples) == 5:
        break

search_results = []

for i, row in enumerate(examples, start=1):
    print(f"\nПример {i}")
    print("Запрос:", row["question1"])
    print("Дубликат по разметке validation:", row["question2"])
    print(
        "Этот дубликат есть среди кандидатов train:",
        row["question2"].strip() in train_questions,
    )

    result = find_duplicates(row["question1"])
    search_results.append(result)
    display(result)

Уникальных вопросов: 493854

Пример 1
Запрос: Is there a reason why we should travel alone?
Дубликат по разметке validation: What are some reasons to travel alone?
Этот дубликат есть среди кандидатов train: True


Поиск дубликатов:   0%|          | 0/15433 [00:00<?, ?it/s]

,question,duplicate_score
0,Why should I travel alone?,0.960510
1,What are some reasons to travel alone?,0.953880
2,If the Indian government has decided to demone...,0.928603
3,Should I travel alone?,0.925836
4,Why does 500 and 1000 Rs notes banned by GOI a...,0.906612



Пример 2
Запрос: I am 25 year old guy and never had a girlfriend. Is this weird?
Дубликат по разметке validation: I am 25 years old. I have never had a girlfriend. Is something wrong with me?
Этот дубликат есть среди кандидатов train: True


Поиск дубликатов:   0%|          | 0/15433 [00:00<?, ?it/s]

...

Поиск дубликатов:   0%|          | 0/15433 [00:00<?, ?it/s]

,question,duplicate_score
0,What should I do in a weekend layover in Hong ...,0.943504
1,What should I do in Hong Kong?,0.935817
2,Why does 500 and 1000 Rs notes banned by GOI a...,0.902610
3,If the Indian government has decided to demone...,0.871037
4,I'm planing to stop at Hong kong for a day. An...,0.864081



Пример 5
Запрос: Which is the best gaming laptop under 40k?
Дубликат по разметке validation: Which is the best gaming laptop under 40,000 rs?
Этот дубликат есть среди кандидатов train: True


Поиск дубликатов:   0%|          | 0/15433 [00:00<?, ?it/s]

,question,duplicate_score
0,"Which is the best gaming laptop under 40,000 rs?",0.981022
1,What is best gaming laptop in the range Rs 40-...,0.945197
2,Can you suggest a good laptop within ₹40k?,0.925701
3,Which laptop is best under 40000 INR?,0.924959
4,What is the best laptop to buy up to Rs 40000 ...,0.922273


### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

### Task 4 (Bonus, 0.5 point):
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

In [ ]:
<A whole lot of YOUR CODE HERE>

### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

### Task 5 (Bonus, 1.5 points):
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

In [ ]:
<A whole lot of YOUR CODE HERE>